In [0]:
# Loading the dataset and displaying the top 10 rows
df_products = spark.table("project.bronze.bronze_products")

display(df_products.head(10))

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_timestamp
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,2026-03-09T13:55:52.301Z
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,2026-03-09T13:55:52.301Z
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,2026-03-09T13:55:52.301Z
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,2026-03-09T13:55:52.301Z
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,2026-03-09T13:55:52.301Z
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11,2026-03-09T13:55:52.301Z
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44,2026-03-09T13:55:52.301Z
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40,2026-03-09T13:55:52.301Z
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17,2026-03-09T13:55:52.301Z
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12,2026-03-09T13:55:52.301Z


### Keep only required columns:

In [0]:
df_products_required = df_products.select(
            "product_id",
            "product_category_name",
            "ingestion_timestamp"
        )

display(df_products_required.head(5))

product_id,product_category_name,ingestion_timestamp
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,2026-03-09T13:55:52.301Z
3aa071139cb16b67ca9e5dea641aaa2f,artes,2026-03-09T13:55:52.301Z
96bd76ec8810374ed1b65e291975717f,esporte_lazer,2026-03-09T13:55:52.301Z
cef67bcfe19066a932b7673e239eb23d,bebes,2026-03-09T13:55:52.301Z
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,2026-03-09T13:55:52.301Z


In [0]:
# Checking the schema for any mismatch
df_products_required.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
# calculating the null values
{col: df_products_required.filter(df_products_required[col].isNull()).count() for col in df_products_required.columns}

{'product_id': 0, 'product_category_name': 610, 'ingestion_timestamp': 0}

### Observation: `product_category_name` is in portugese and has 610 missing values

Translate them to english first and fill the missing values with category `unknown`

In [0]:
df_translation = spark.table("project.bronze.bronze_product_category_translation").drop("ingestion_timestamp")
display(df_translation.limit(5))

product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor


In [0]:
df_products_with_english = df_products_required.join(
    df_translation,
    on="product_category_name",
    how="left"
)


display(df_products_with_english.head(5))

product_category_name,product_id,ingestion_timestamp,product_category_name_english
perfumaria,1e9e8ef04dbcff4541ed26657ea517e5,2026-03-09T13:55:52.301Z,perfumery
artes,3aa071139cb16b67ca9e5dea641aaa2f,2026-03-09T13:55:52.301Z,art
esporte_lazer,96bd76ec8810374ed1b65e291975717f,2026-03-09T13:55:52.301Z,sports_leisure
bebes,cef67bcfe19066a932b7673e239eb23d,2026-03-09T13:55:52.301Z,baby
utilidades_domesticas,9dc1a7de274444849c219cff195d0b71,2026-03-09T13:55:52.301Z,housewares


### Fill the Null Values with category `unknown`

In [0]:
from pyspark.sql.functions import col, when

df_products_new = df_products_with_english.withColumn(
            "product_category_name_english",
            when(
                col("product_category_name_english").isNull(),
                "unknown_category"
            ).otherwise(col("product_category_name_english")))

# calculating the null values
{col: df_products_new.filter(df_products_new[col].isNull()).count() for col in df_products_new.columns}

{'product_category_name': 610,
 'product_id': 0,
 'ingestion_timestamp': 0,
 'product_category_name_english': 0}

### Clean the category names and drop the duplicates

In [0]:
from pyspark.sql.functions import col, lower, trim

# Clean the English category name (lowercase and trim)
df_products_new = df_products_new.withColumn(
    "product_category_name_english",
    lower(trim(col("product_category_name_english")))
)

# Drop duplicate products by product_id
df_products_new = df_products_new.dropDuplicates(["product_id"])


In [0]:
df_products_new.show(5)

+---------------------+--------------------+--------------------+-----------------------------+
|product_category_name|          product_id| ingestion_timestamp|product_category_name_english|
+---------------------+--------------------+--------------------+-----------------------------+
|   ferramentas_jardim|e1d1d22e9f8122a4e...|2026-03-09 13:55:...|                 garden_tools|
|        esporte_lazer|ce5b91848b91118da...|2026-03-09 13:55:...|               sports_leisure|
|           brinquedos|1c6fb703c624b381a...|2026-03-09 13:55:...|                         toys|
|            papelaria|4e04ffb7dd3739ecf...|2026-03-09 13:55:...|                   stationery|
|   ferramentas_jardim|0992c6cba95a13bfa...|2026-03-09 13:55:...|                 garden_tools|
+---------------------+--------------------+--------------------+-----------------------------+
only showing top 5 rows


In [0]:
# Drop the original category name column
df_products_final = df_products_new.drop("product_category_name")

display(df_products_final.head(4))

product_id,ingestion_timestamp,product_category_name_english
e1d1d22e9f8122a4ec1533b032c12562,2026-03-09T13:55:52.301Z,garden_tools
ce5b91848b91118daffb3af53b747475,2026-03-09T13:55:52.301Z,sports_leisure
1c6fb703c624b381a20f21f757694866,2026-03-09T13:55:52.301Z,toys
4e04ffb7dd3739ecfc37de8927dd586c,2026-03-09T13:55:52.301Z,stationery


In [0]:
# saving the cleaned data
df_products_final.write.format('delta').mode('overwrite').saveAsTable("project.silver.silver_product")